 # **Scraping Ulasan Aplikasi Livin' by Mandiri**
oleh : Errie Tri Armawan (2026)
## Tentang Proyek
Notebook ini adalah tahap pengumpulan data untuk proyek **analisis sentimen** ulasan pengguna aplikasi **Livin' by Mandiri** di Google Play Store. Tujuan akhir proyek adalah membangun model yang dapat mengklasifikasikan ulasan ke dalam kelas sentimen (negatif, netral, positif).

Proyek ini terdiri dari dua notebook:
1. **Scraping** (notebook ini): mengambil ulasan dan menyimpannya sebagai dataset CSV.
2. **Pelatihan model**: pelabelan, ekstraksi fitur, pelatihan beberapa skema model, evaluasi, dan inferensi.

## Sumber dan Spesifikasi Data
- **Sumber:** Google Play Store, aplikasi Livin' by Mandiri (`id.bmri.livin`)
- **Bahasa dan wilayah:** Indonesia (`lang='id'`, `country='id'`)
- **Target jumlah:** 15.000 ulasan
- **Metode:** library `google-play-scraper` (tanpa login, hanya data ulasan yang tampil publik)
- **Tanggal pengambilan:** 12 Oktober 2026

## Alur di Notebook Ini
*   Menghubungkan Google Drive dan menyiapkan path
*   Instalasi dan import library
*   Konfigurasi scraping
*   Proses scraping per bintang
*   Menyimpanan ke CSV




## **Menghubungkan Google Drive dan Menyiapkan Path**
Google Drive dihubungkan (*mount*) agar dataset hasil scraping langsung tersimpan di Drive. Penyimpanan di Drive dipilih karena file di `/content` Colab akan hilang ketika runtime terputus, sedangkan proses scraping membutuhkan waktu yang cukup lama.

Path folder penyimpanan dan nama file keluaran juga didefinisikan di awal agar mudah diubah dan konsisten digunakan di seluruh notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

FOLDER = '/content/drive/MyDrive/Colab Notebooks/submission_sentimen_analisis'
os.makedirs(FOLDER, exist_ok=True)

PATH_CSV = os.path.join(FOLDER, 'ulasan_aplikasi_livin.csv')

print('Folder   :', FOLDER)
print('Output   :', PATH_CSV)


Folder   : /content/drive/MyDrive/Colab Notebooks/submission_sentimen_analisis
Output   : /content/drive/MyDrive/Colab Notebooks/submission_sentimen_analisis/ulasan_aplikasi_livin.csv


## **Instalasi dan Import Library**
Library yang digunakan:
- `google-play-scraper`: mengambil ulasan dari Google Play Store
- `pandas`: menyusun data dalam bentuk tabel dan menyimpannya ke CSV
- `time`: memberi jeda antar permintaan agar tidak membebani server

In [3]:
# Instalasi library
!pip install google-play-scraper -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.1 MB/s eta 0:00:00


In [4]:
# =============================================================
# Import Library
# =============================================================
import time
import pandas as pd
from google_play_scraper import Sort, reviews

## **Konfigurasi Scraping**
Pengaturan yang dipakai:
- **`APP_ID`**: ID aplikasi, diambil dari URL halaman aplikasi di Play Store.
- **`TARGET_PER_BINTANG`**: jumlah ulasan yang diambil untuk tiap rating (1-5), total 15.000. Data diambil per bintang, bukan acak dari seluruh ulasan, karena rating 3 jumlahnya biasanya jauh lebih sedikit. Dengan cara ini, kelas dengan ulasan sedikit tetap memiliki data yang cukup.
- **`BATCH`** dan **`JEDA`**: jumlah ulasan per permintaan dan jeda antar permintaan.

> **Catatan:** karena pengambilan dibuat per bintang, komposisi dataset tidak mencerminkan sebaran rating asli di Play Store. Hal ini dicatat sebagai keterbatasan.

In [12]:
# =============================================================
# Konfigurasi
# =============================================================
APP_ID = 'id.bmri.livin'

# Target per bintang (total 15.000)
# Negatif = bintang 1-2, Netral = bintang 3, Positif = bintang 4-5
TARGET_PER_BINTANG = {1: 2500, 2: 2500, 3: 5000, 4: 2500, 5: 2500}

BATCH = 200          # jumlah ulasan per permintaan
JEDA = 1             # detik, jeda antar permintaan

## **Proses Scraping**
Untuk setiap rating, ulasan diambil bertahap (per `BATCH`) dengan memanfaatkan `continuation_token` sebagai penanda halaman terakhir, sampai target tercapai atau ulasan habis. Ulasan diurutkan dari yang terbaru (`Sort.NEWEST`). Jika terjadi error, proses menunggu beberapa detik lalu mencoba lagi.

### Mengapa menggunakan `reviews`, bukan `reviews_all`?
Pada modul *Studi Kasus Implementasi Klasifikasi Teks pada NLP: Sentimen Analisis Review APK Play Store*, fungsi `reviews_all` digunakan dengan parameter `count=1000`. Hasilnya, **seluruh ulasan aplikasi tetap diambil (213.448 baris)** dan parameter `count` diabaikan. Hal ini memakan waktu lama dan meningkatkan risiko pembatasan akses dari server.

Karena itu pada proyek ini digunakan fungsi `reviews`, dengan alasan:
1. **Jumlah data dapat dibatasi** sesuai `count`, sehingga hanya data yang dibutuhkan yang diambil.
2. **Dapat difilter per rating** (`filter_score_with`), sehingga komposisi kelas dapat dikendalikan.
3. **Pengambilan bertahap** dengan `continuation_token`, sehingga lebih terkendali dan dapat dihentikan atau dilanjutkan.

In [13]:
# =============================================================
# Scraping per Bintang
# =============================================================
semua_ulasan = []

for bintang, target in TARGET_PER_BINTANG.items():
    terkumpul = []
    token = None

    while len(terkumpul) < target:
        try:
            hasil, token = reviews(
                APP_ID,
                lang='id',
                country='id',
                sort=Sort.NEWEST,
                count=min(BATCH, target - len(terkumpul)),
                filter_score_with=bintang,
                continuation_token=token
            )
        except Exception as e:
            print(f'Bintang {bintang}: error ({e}), tunggu lalu coba lagi...')
            time.sleep(10)
            continue

        if not hasil:      # tidak ada ulasan lagi
            break

        terkumpul.extend(hasil)
        print(f'Bintang {bintang}: {len(terkumpul)}/{target}')
        time.sleep(JEDA)

        if token is None:  # sudah halaman terakhir
            break

    semua_ulasan.extend(terkumpul)

print('Total terkumpul:', len(semua_ulasan))


Bintang 1: 200/2500
Bintang 1: 400/2500
Bintang 1: 600/2500
Bintang 1: 800/2500
Bintang 1: 1000/2500
Bintang 1: 1200/2500
Bintang 1: 1400/2500
Bintang 1: 1600/2500
Bintang 1: 1800/2500
Bintang 1: 2000/2500
Bintang 1: 2200/2500
Bintang 1: 2400/2500
Bintang 1: 2600/2500
Bintang 2: 200/2500
Bintang 2: 400/2500
Bintang 2: 600/2500
Bintang 2: 800/2500
Bintang 2: 1000/2500
Bintang 2: 1200/2500
Bintang 2: 1400/2500
Bintang 2: 1600/2500
Bintang 2: 1800/2500
Bintang 2: 2000/2500
Bintang 2: 2200/2500
Bintang 2: 2400/2500
Bintang 2: 2600/2500
Bintang 3: 200/5000
Bintang 3: 400/5000
Bintang 3: 600/5000
Bintang 3: 800/5000
Bintang 3: 1000/5000
Bintang 3: 1200/5000
Bintang 3: 1400/5000
Bintang 3: 1600/5000
Bintang 3: 1800/5000
Bintang 3: 2000/5000
Bintang 3: 2200/5000
Bintang 3: 2400/5000
Bintang 3: 2600/5000
Bintang 3: 2800/5000
Bintang 3: 3000/5000
Bintang 3: 3200/5000
Bintang 3: 3400/5000
Bintang 3: 3600/5000
Bintang 3: 3800/5000
Bintang 3: 4000/5000
Bintang 3: 4200/5000
Bintang 3: 4400/5000
Bint

## **Menyimpan Hasil Scraping**
Hasil scraping diubah menjadi `DataFrame` dengan `pandas`, lalu disimpan sebagai file CSV (`ulasan_aplikasi.csv`) di Google Drive.

**Kolom yang disimpan:**
- `reviewId`: ID unik ulasan, dipakai untuk mendeteksi duplikat pada tahap preprocessing
- `content`: teks ulasan
- `score`: rating 1-5, dasar pelabelan sentimen
- `at`: waktu ulasan dibuat

Kolom yang memuat identitas pengguna (`userName`, `userImage`) tidak disimpan demi menjaga privasi.

File ini dibaca oleh notebook  pelatihan model.


In [14]:
# Mengubah hasil scraping menjadi DataFrame
df = pd.DataFrame(semua_ulasan)

# Menyimpan kolom yang penting saja (kolom identitas pengguna tidak disimpan)
df = df[['reviewId', 'content', 'score', 'at']]

# Menyimpan hasil scraping ke Google Drive
df.to_csv(PATH_CSV, index=False)

# Mengecek jumlah data, sebaran rating, dan jumlah duplikat
print(df.shape)
print(df['score'].value_counts().sort_index())

(15400, 4)
score
1    2600
2    2600
3    5000
4    2600
5    2600
Name: count, dtype: int64
